<a href="https://colab.research.google.com/github/ruchit0002/Edgesense/blob/main/Edgesense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import time

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
file = list(uploaded.keys())[0]

df = pd.read_csv(file)
df.columns = df.columns.str.strip()

print("Columns:", df.columns)
df.head()

In [ ]:
text_col = None
label_col = None

for col in df.columns:
    sample = str(df[col].iloc[0])

    if isinstance(sample, str) and len(sample) > 20:
        text_col = col
    else:
        label_col = col

if text_col is None:
    text_col = df.columns[0]

if label_col is None:
    label_col = df.columns[1]

print("Text column:", text_col)
print("Label column:", label_col)

In [ ]:
def convert_label(x):
    try:
        return int(x)
    except:
        x = str(x).lower()
        if "pos" in x:
            return 1
        elif "neg" in x:
            return 0
        else:
            return None

df[label_col] = df[label_col].apply(convert_label)

df = df.dropna(subset=[label_col])
df[label_col] = df[label_col].astype(int)

In [ ]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

model.eval()

print("Model loaded.")

In [ ]:
correct = 0
total = 0
total_time = 0

for i in range(min(100, len(df))):

    text = str(df[text_col].iloc[i])
    label = int(df[label_col].iloc[i])

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    start = time.time()

    with torch.no_grad():
        outputs = model(**inputs)

    end = time.time()

    total_time += (end - start)

    pred = torch.argmax(outputs.logits).item()

    if pred == label:
        correct += 1

    total += 1

accuracy = correct / total

print("=== Results ===")
print("Samples tested:", total)
print("Accuracy:", round(accuracy*100,2), "%")
print("Average Inference Time:", round((total_time/total)*1000,2), "ms")

In [ ]:
while True:
    user_input = input("Enter text (or type exit): ")

    if user_input.lower() == "exit":
        break

    inputs = tokenizer(user_input, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits).item()

    result = "Positive" if pred == 1 else "Negative"

    print("Prediction:", result)